In [ ]:
!pip install -q pandas openpyxl

import pandas as pd

file_paths = [
    '_aligned_analysis_הודיה.xlsx',
    'aligned_analysis_זאב_complete_all_referents.xlsx',
    'LENA_MERGED.xlsx',
    'MERAV_MERGED.xlsx'
]

possible_sentence_cols = ['full_raw_response', 'Full_Raw_Sentence']

file_unique_sentences = {}
all_sentences = set()

# Read files and collect unique sentences
for file in file_paths:
    try:
        df = pd.read_excel(file)

        # Identify the relevant sentence column
        sentence_col = next((col for col in df.columns if col in possible_sentence_cols), None)

        if sentence_col is None:
            print(f"⚠️ Could not find sentence column in '{file}'. Found columns: {list(df.columns)}")
            continue

        # Clean text and collect unique non-empty sentences
        sentences = set(df[sentence_col].dropna().astype(str).str.strip())
        file_unique_sentences[file] = sentences
        all_sentences.update(sentences)

    except FileNotFoundError:
        print(f"❌ File not found: '{file}'. Make sure it is uploaded to the main Colab directory.")

total_unique_count = len(all_sentences)

# Calculate and display percentages
print(f"=== SUMMARY ===")
print(f"Total Unique Sentences Across All Files: {total_unique_count}\n")

results = []
for file, sentences in file_unique_sentences.items():
    count = len(sentences)
    pct = (count / total_unique_count * 100) if total_unique_count > 0 else 0
    results.append({
        'File Name': file,
        'Unique Sentences': count,
        'Coverage (%)': f"{pct:.2f}%"
    })

summary_df = pd.DataFrame(results)
summary_df

=== SUMMARY ===
Total Unique Sentences Across All Files: 4998



,File Name,Unique Sentences,Coverage (%)
0,_aligned_analysis_הודיה.xlsx,1104,22.09%
1,aligned_analysis_זאב_complete_all_referents.xlsx,337,6.74%
2,LENA_MERGED.xlsx,1451,29.03%
3,MERAV_MERGED.xlsx,2147,42.96%


In [ ]:
!pip install -q pandas openpyxl

import pandas as pd

file_paths = [
    '_aligned_analysis_הודיה.xlsx',
    'aligned_analysis_זאב_complete_all_referents.xlsx',
    'LENA_MERGED.xlsx',
    'MERAV_MERGED.xlsx'
]

possible_sentence_cols = ['full_raw_response', 'Full_Raw_Sentence', 'full_raw_sentence']
possible_prompt_cols = ['target_word', 'Prompt_Word', 'prompt_word', 'Target_Word']

combined_records = []

# Read files and collect unique (prompt_word, sentence) pairs
for file in file_paths:
    try:
        df = pd.read_excel(file)

        sent_col = next((c for c in df.columns if c in possible_sentence_cols), None)
        prompt_col = next((c for c in df.columns if c in possible_prompt_cols), None)

        if sent_col and prompt_col:
            temp = df[[prompt_col, sent_col]].dropna().copy()
            temp.columns = ['prompt_word', 'sentence']
            temp['prompt_word'] = temp['prompt_word'].astype(str).str.strip()
            temp['sentence'] = temp['sentence'].astype(str).str.strip()

            combined_records.append(temp)

    except FileNotFoundError:
        print(f"❌ File not found: '{file}'")

# Merge into a single deduplicated dataset of unique sentences
df_all = pd.concat(combined_records, ignore_index=True)
unique_sentences_df = df_all.drop_duplicates(subset=['prompt_word', 'sentence'])

total_unique_sentences = len(unique_sentences_df)

# Count unique sentences per prompt word and calculate % share
prompt_summary = (
    unique_sentences_df['prompt_word']
    .value_counts()
    .reset_index()
)
prompt_summary.columns = ['Prompt Word', 'Sentence Count']

# Calculate percentage share (sums to 100%)
prompt_summary['Percentage Share'] = (
    (prompt_summary['Sentence Count'] / total_unique_sentences) * 100
).round(2).astype(str) + '%'

print(f"Total Unique Sentences: {total_unique_sentences}\n")

prompt_summary

Total Unique Sentences: 5061



,Prompt Word,Sentence Count,Percentage Share
0,טלוויזיה,298,5.89%
1,דרך,288,5.69%
2,לתת,254,5.02%
3,הר,248,4.9%
4,מכונית,213,4.21%
5,לשבור,206,4.07%
6,שמחה,201,3.97%
7,אוכל,200,3.95%
8,ים,193,3.81%
9,סיפור,193,3.81%


In [ ]:
!pip install -q pandas openpyxl

import pandas as pd
import numpy as np
import unicodedata

def remove_nikud(text):
    if not isinstance(text, str):
        return text
    normalized = unicodedata.normalize('NFD', text)
    stripped = ''.join(c for c in normalized if unicodedata.category(c) != 'Mn')
    return unicodedata.normalize('NFC', stripped).strip()

file_paths = [
    '_aligned_analysis_הודיה.xlsx',
    'aligned_analysis_זאב_complete_all_referents.xlsx',
    'LENA_MERGED.xlsx',
    'MERAV_MERGED.xlsx'
]

COLUMN_MAP = {
    'questionn': 'Questionnaire_ID',
    'questionnaire_id': 'Questionnaire_ID',
    'Questionnaire': 'Questionnaire_ID',

    'target_word': 'Prompt_Word',
    'prompt_word': 'Prompt_Word',
    'Target_Word': 'Prompt_Word',
    'Prompt_Word': 'Prompt_Word',

    'full_raw_response': 'Full_Raw_Sentence',
    'full_raw_sentence': 'Full_Raw_Sentence',
    'Full_Raw_Sentence': 'Full_Raw_Sentence',

    'referent': 'Referent',
    'Referent': 'Referent',
    'meaningVal': 'MeaningVal',
    'MeaningVal': 'MeaningVal',
    'Original_ID': 'Original_ID',
    'original_id': 'Original_ID'
}

dfs = []

for file in file_paths:
    try:
        df = pd.read_excel(file)
        df = df.rename(columns=COLUMN_MAP)
        df['source_student_file'] = file
        dfs.append(df)
    except FileNotFoundError:
        print(f"❌ File not found: {file}")

df_all = pd.concat(dfs, ignore_index=True)

# Drop rows missing sentence or prompt word
df_all = df_all.dropna(subset=['Prompt_Word', 'Full_Raw_Sentence']).copy()

# Clean string values & STRIP NIKUD
df_all['Prompt_Word'] = df_all['Prompt_Word'].astype(str).apply(remove_nikud)
df_all['Full_Raw_Sentence'] = df_all['Full_Raw_Sentence'].astype(str).apply(remove_nikud)

df_all = df_all.fillna("")

# ----------------- PROMPT-BASED SPLIT ----------------- #
unique_sentence_map = df_all.drop_duplicates(subset=['Prompt_Word', 'Full_Raw_Sentence'])
prompt_counts = unique_sentence_map['Prompt_Word'].value_counts()

total_unique_sentences = len(unique_sentence_map)
target_test_count = int(0.15 * total_unique_sentences)
target_val_count = int(0.15 * total_unique_sentences)

shuffled_prompts = prompt_counts.sample(frac=1, random_state=42)

test_prompts, val_prompts, train_prompts = [], [], []
current_test, current_val = 0, 0

for prompt, count in shuffled_prompts.items():
    if current_test < target_test_count:
        test_prompts.append(prompt)
        current_test += count
    elif current_val < target_val_count:
        val_prompts.append(prompt)
        current_val += count
    else:
        train_prompts.append(prompt)

# Filter full rows into train/val/test splits
df_train = df_all[df_all['Prompt_Word'].isin(train_prompts)].copy()
df_val = df_all[df_all['Prompt_Word'].isin(val_prompts)].copy()
df_test = df_all[df_all['Prompt_Word'].isin(test_prompts)].copy()

# Save cleaned CSV files
df_train.to_csv('train.csv', index=False, encoding='utf-8-sig')
df_val.to_csv('val.csv', index=False, encoding='utf-8-sig')
df_test.to_csv('test.csv', index=False, encoding='utf-8-sig')

print("✅ Complete! Nikud stripped, prompt words unified, and CSVs saved cleanly.")
print(f"Train rows: {len(df_train)} | Val rows: {len(df_val)} | Test rows: {len(df_test)}")

✅ Complete! Nikud stripped, prompt words unified, and CSVs saved cleanly.
Train rows: 18475 | Val rows: 4296 | Test rows: 5261


Row Count

In [ ]:
import pandas as pd

train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('val.csv')
test_df = pd.read_csv('test.csv')

total_rows = len(train_df) + len(val_df) + len(test_df)

print("=================== אחוזי החלוקה הכלליים ===================")
print(f"סך כל השורות בנתונים: {total_rows}\n")

print(f"• Train set: {len(train_df):,} שורות | {len(train_df)/total_rows*100:.2f}%")
print(f"• Val set:   {len(val_df):,} שורות | {len(val_df)/total_rows*100:.2f}%")
print(f"• Test set:  {len(test_df):,} שורות | {len(test_df)/total_rows*100:.2f}%\n")

def print_file_percentages(df, set_name):
    print(f"------------------- הרכב הקבצים ב- {set_name} -------------------")
    file_counts = df['source_student_file'].value_counts()
    set_total = len(df)

    for file_name, count in file_counts.items():
        pct_of_set = (count / set_total) * 100

        short_name = file_name.replace('.xlsx', '').replace('_aligned_analysis_', '').replace('aligned_analysis_', '')
        print(f"  • {short_name}: {count:,} שורות ({pct_of_set:.2f}% מסך ה-{set_name})")
    print()

print_file_percentages(train_df, "Train")
print_file_percentages(val_df, "Validation")
print_file_percentages(test_df, "Test")

=================== אחוזי החלוקה הכלליים ===================
סך כל השורות בנתונים: 28032

• Train set: 18,475 שורות | 65.91%
• Val set:   4,296 שורות | 15.33%
• Test set:  5,261 שורות | 18.77%

------------------- הרכב הקבצים ב- Train -------------------
  • LENA_MERGED: 7,027 שורות (38.04% מסך ה-Train)
  • MERAV_MERGED: 6,217 שורות (33.65% מסך ה-Train)
  • הודיה: 3,825 שורות (20.70% מסך ה-Train)
  • זאב_complete_all_referents: 1,406 שורות (7.61% מסך ה-Train)

------------------- הרכב הקבצים ב- Validation -------------------
  • MERAV_MERGED: 1,830 שורות (42.60% מסך ה-Validation)
  • LENA_MERGED: 1,726 שורות (40.18% מסך ה-Validation)
  • הודיה: 542 שורות (12.62% מסך ה-Validation)
  • זאב_complete_all_referents: 198 שורות (4.61% מסך ה-Validation)

------------------- הרכב הקבצים ב- Test -------------------
  • LENA_MERGED: 2,186 שורות (41.55% מסך ה-Test)
  • הודיה: 1,706 שורות (32.43% מסך ה-Test)
  • MERAV_MERGED: 794 שורות (15.09% מסך ה-Test)
  • זאב_complete_all_referents: 575 שורות (